# 02 — Bias Analysis
---

## Table of Contents

This notebook loads the NovaCred credit clean dataset and performs a bias data audit showing the bias patterns present in this dataset.

**Input:** `data/processed/applications_analysis_ready.csv` 

**Outputs:**  
- `xxxxx.csv` (xxxx)  

The notebook covers the disparate impact (DI) ratio for gender and interprets it agains the four-fifths rule, identifies the proxy discriminations, investigate agd-biased bias patterns and explores the interaction effects xxxxx.

In [18]:
# Now import
import pandas as pd
from pathlib import Path
import sys
import os

In [19]:
import numpy as np

---
## 0. Load the data 
---

In [20]:
# Resolve root regardless of where the notebook is run from
cwd = Path.cwd()
root = cwd.parent if cwd.name == "notebooks" else cwd

# Build path to the cleaned file saved in the previous notebook
clean_path = root / "data" / "processed" / "applications_clean.csv"

# Load into dataframe
df = pd.read_csv(clean_path)

# Quick sanity check
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (500, 20)
Columns: ['_id', 'spending_behavior', 'processing_timestamp', 'applicant_info.full_name', 'applicant_info.email', 'applicant_info.ssn', 'applicant_info.ip_address', 'applicant_info.gender', 'applicant_info.date_of_birth', 'applicant_info.zip_code', 'financials.annual_income', 'financials.credit_history_months', 'financials.debt_to_income', 'financials.savings_balance', 'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate', 'decision.approved_amount', 'ssn_is_duplicate']


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,ssn_is_duplicate
0,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230.0,102000.0,37.0,0.42,0.0,False,high_dti_ratio,NaN,NaN,NaN,False
1,app_002,"[{'category': 'Education', 'amount': 533}]",NaN,Kevin Roberts,kevin.roberts9@protonmail.com,992-61-4010,172.19.95.144,Male,1999-08-01,10020.0,41000.0,5.0,0.36,18200.0,False,algorithm_risk_score,NaN,NaN,NaN,False
2,app_003,"[{'category': 'Healthcare', 'amount': 450}]",NaN,Lisa Gonzalez,lisa.gonzalez51@yahoo.com,833-33-5929,172.21.35.195,Female,1982-08-24,90213.0,65000.0,74.0,0.43,7090.0,True,NaN,NaN,3.4,76000.0,False
3,app_004,"[{'category': 'Transportation', 'amount': 329}...",NaN,Karen Nelson,karen.nelson35@outlook.com,486-50-5539,172.31.79.76,Female,1995-02-28,90217.0,69000.0,9.0,0.41,10327.0,False,high_dti_ratio,NaN,NaN,NaN,False
4,app_005,"[{'category': 'Insurance', 'amount': 585}]",NaN,Christine Mitchell,christine.mitchell3@outlook.com,400-91-8156,172.25.44.173,Female,1960-06-19,90296.0,39000.0,76.0,0.06,15011.0,False,algorithm_risk_score,NaN,NaN,NaN,False


---
## 01. Gender disparate impact
---

In this section, we investigate whether loan approvals show gender-based disparate impact. The goal is to determine if `decision.loan_approved` produces unequal outcomes across gender groups - `applicant_info.gender`. 

First we start by computing the aprovable rates of each gender. 
Then we defined with is the advantage group and how is a disadvantage group. 
Based on that we compute the Adverse Impact Ratio (AIR), that is the ratio between the approal rate of the disadvantage group and the advantage group. 
By the 4/5ths Rule (AIR= 0.8) we will take conclusions. 

In [21]:
#Compute the approvable rates 

gender_approved_rates = df.groupby("applicant_info.gender")["decision.loan_approved"].mean()
gender_approved_rates

applicant_info.gender
Female    0.505976
Male      0.659919
Name: decision.loan_approved, dtype: float64

Since provables rates are lower to female we count the feminine gender as the disadvantaged group. 

In [22]:
#Compute the AIR
AIR = gender_approved_rates["Female"]/gender_approved_rates ["Male"]
AIR

np.float64(0.7667245129909809)

Since AIR = 0.767 is below the 4/5's Rule (0.8) but its close, being on the borderline is worth of investigation. Butis the difference in approval rates between men and women real, or could it just be a coincidence?
Since AIR alone does not tell us if the disparaty of the approvable rates by gender are real or a random variation of our data. Is needed a  Statistical significance test to understand how much data you have, how likely is it that this gap happened randomly? 

In [23]:
#Creates a frequency table counting how many Males/Females were approved/rejected

freq_table = pd.crosstab(df["applicant_info.gender"], df["decision.loan_approved"])
freq_table

# Extracts observable values
observed_values = freq_table.values 

# Computing total approved across everyone, total applicants per gender and the grand total 
col_totals = observed_values.sum(axis=0, keepdims=True)
row_totals = observed_values.sum(axis=1, keepdims=True)
grand_total = observed_values.sum()

# The "no gender bias" baseline we compare against
expected = row_totals * col_totals /grand_total

#Chi2 test and p-value and check for statistical significanse
chi2 = ((observed_values - expected) ** 2 / expected).sum()
p_value = 1 - np.exp(-chi2 / 2)

if p_value < 0.05:
    print("Result: statistically significant")
else:
    print("Result: not statistically significant")

Result: not statistically significant


The AIR of 0.767 falls slightly below the 0.80 threshold, flagging a potential gender disparity. However, the result is not statistically significant despite a sample of 500 applicants. This suggests the disparity is modest rather than hidden by sample size. There is no conclusive evidence of disparate impact, but the approval gap warrants continued monitoring.

---
## 02. Age-Based discrimination patterns
---

---
## 03. Proxy variables for protected attributes
---

---
## 04. Interaction effects between attributes
---

---
## 0.5 Conclusion
---